# Preprocess Dosage Sensitivity Region and Gene Data
* Dosage sensitivity data from [ClinGen](https://search.clinicalgenome.org/kb/downloads)
* Used the Archived 2024-12-20 dosage sensitivity [gene](https://ftp.clinicalgenome.org/archive/20241220/ClinGen_haploinsufficiency_gene_GRCh38.bed) and [region](https://ftp.clinicalgenome.org/archive/20241220/ClinGen_haploinsufficiency_gene_GRCh38.bed) files with haploinsufficiency (HI) and triplosensitivity (TS) scores equal to 3 ("sufficient evidence" tier)

In [1]:
import pandas as pd
from tqdm.notebook import tqdm
import plotly.graph_objects as go

In [2]:
dosage_gene_path = "../clingen_dosage/ClinGen_gene_curation_list_GRCh38.tsv"
dosage_region_path = "../clingen_dosage/ClinGen_region_curation_list_GRCh38.tsv"

In [3]:
dosage_gene_df = (
    pd.read_csv(dosage_gene_path, delimiter='\t', header=5)
    .rename(columns=lambda col: col.strip('#')) #remove comment character from header
)
dosage_region_df = (
    pd.read_csv(dosage_region_path, delimiter='\t', header=5)
    .rename(columns=lambda col: col.strip('#'))
)

In [4]:
dosage_gene_df['feature_type'] = 'gene'
dosage_region_df['feature_type'] = 'region'

dosage_gene_df['feature_name'] = dosage_gene_df['Gene Symbol']
dosage_gene_df['feature_id'] = dosage_gene_df['Gene ID']

dosage_region_df['feature_name'] = dosage_region_df['ISCA Region Name']
dosage_region_df['feature_id'] = dosage_region_df['ISCA ID']

dosage_df = pd.concat(
    [dosage_gene_df, dosage_region_df]
)
len(dosage_df)

2073

In [5]:
dosage_df = dosage_df.query("`Genomic Location` != 'tbd'")
len(dosage_df)

2065

In [6]:
dosage_df['chromosome'] = dosage_df['Genomic Location'].str.extract(
    r"chr([\dXY]+)\:"
)

dosage_df['genomic_start'] = dosage_df['Genomic Location'].str.extract(
    r"\:(\d+)\-"
).astype(int)
dosage_df['genomic_stop'] = dosage_df['Genomic Location'].str.extract(
    r"\d+\-(\d+)"
).astype(int)

In [7]:
(
    dosage_df
    .groupby(['feature_type', 'Haploinsufficiency Score', 'Haploinsufficiency Description'])
    .size()
    .unstack(level=0, fill_value=0)
)

,feature_type,gene,region
Haploinsufficiency Score,Haploinsufficiency Description,,
0.0,No evidence available,274,89
1.0,Little evidence for dosage pathogenicity,127,9
2.0,Some evidence for dosage pathogenicity,8,8
3.0,Sufficient evidence for dosage pathogenicity,380,48
30.0,Gene associated with autosomal recessive phenotype,731,3
40.0,Dosage sensitivity unlikely,35,352


In [8]:
(
    dosage_df
    .groupby(['feature_type', 'Triplosensitivity Score', 'Triplosensitivity Description'])
    .size()
    .unstack(level=0, fill_value=0)
)

,feature_type,gene,region
Triplosensitivity Score,Triplosensitivity Description,,
0,No evidence available,1267,367
1,Little evidence for dosage pathogenicity,8,19
2,Some evidence for dosage pathogenicity,1,10
3,Sufficient evidence for dosage pathogenicity,2,21
30,Gene associated with autosomal recessive phenotype,4,0
40,Dosage sensitivity unlikely,3,92
Not yet evaluated,Not yet evaluated,269,1


In [11]:
feat_df_hi = (
    dosage_df
    .query("`Haploinsufficiency Score` == 3")
)[['feature_type', 'feature_name', 'feature_id', 'chromosome', 'genomic_start', 'genomic_stop']]
len(feat_df_hi)

428

In [12]:
feat_df_ts = (
    dosage_df
    .query("`Triplosensitivity Score` == '3'") # score is a string type in this table
)[['feature_type', 'feature_name', 'feature_id', 'chromosome', 'genomic_start', 'genomic_stop']]
len(feat_df_ts)

23

In [13]:
# load NCH and ClinVar CNVs
nch_df = pd.read_csv("cnv_data/NCH-microarray-CNVs-cleaned.csv")
clinvar_df = pd.read_csv("cnv_data/ClinVar-CNVs-normalized.csv.gzip", compression='gzip')

# NCH to ClinVar matching with respect to dosage sensitivity

## Matching NCH copy loss CNVs to ClinVar haploinsufficiency overlapping variants

In [39]:
# create subset dataframes for copy gain/loss
nch_df_loss = nch_df.query("copy_number_max < 2")
len(nch_df_loss)

3995

In [40]:
#only keep copy loss ClinVar CNVs
clinvar_df_loss = clinvar_df.query("absolute_copies < 2") 
len(clinvar_df_loss)

18856

In [41]:
# cross reference with all HI features
clinvar_hi_df = (
    clinvar_df_loss
    .merge( 
        right=feat_df_hi,
        how='inner',
        left_on='chr',
        right_on='chromosome',
        suffixes=('__clinvar', '__ts')
    )
)
len(clinvar_hi_df)

420472

In [42]:
# keep partial overlap with HI region
clinvar_hi_df = (
    clinvar_hi_df
    #.query("genomic_start.between(start_38, stop_38)") # overlapping with beginning of region
    #.query("genomic_stop.between(start_38, stop_38)") #overlapping with end of region
    .query("genomic_start.between(start_38, stop_38) | genomic_stop.between(start_38, stop_38)")
)
len(clinvar_hi_df)

14805

In [47]:
clinvar_df_hi_overlap = pd.DataFrame(
    clinvar_hi_df
    .groupby(clinvar_df.columns.to_list())
    ['feature_name']
    .nunique()
    .reset_index()
)
len(clinvar_df_hi_overlap)

4529

In [53]:
nch_clinvar_hi_match_df = (
    nch_df_loss.merge(
        right=clinvar_df_hi_overlap,
        left_on='chromosome',
        right_on='chr',
        suffixes=('__nch', '__clinvar'),
        how='inner'
    )
    
)

overlap_start_query = "start_38__nch.between(start_38__clinvar, stop_38__clinvar)"
overlap_stop_query = "start_38__nch.between(start_38__clinvar, stop_38__clinvar)"
overlap_or_query = '|'.join([overlap_start_query, overlap_stop_query])

nch_clinvar_hi_match_df = (
    nch_clinvar_hi_match_df
    .query(overlap_or_query)
)

len(nch_clinvar_hi_match_df)

81083

### Overlap Jaccard Score for NCH copy loss variants vs ClinVar copy loss / HI variants

In [65]:
nch_clinvar_hi_match_df['start_outer'] = nch_clinvar_hi_match_df[
    ['start_38__nch', 'start_38__clinvar']
    ].min(axis=1)

nch_clinvar_hi_match_df['start_inner'] = nch_clinvar_hi_match_df[
    ['start_38__nch', 'start_38__clinvar']
    ].max(axis=1)

nch_clinvar_hi_match_df['stop_inner'] = nch_clinvar_hi_match_df[
    ['stop_38__nch', 'stop_38__clinvar']
    ].min(axis=1)

nch_clinvar_hi_match_df['stop_outer'] = nch_clinvar_hi_match_df[
    ['stop_38__nch', 'stop_38__clinvar']
    ].max(axis=1)

nch_clinvar_hi_match_df['intersection_length'] = (
    nch_clinvar_hi_match_df['stop_inner'] - nch_clinvar_hi_match_df['start_inner']
)

nch_clinvar_hi_match_df['union_length'] = (
    nch_clinvar_hi_match_df['stop_outer'] - nch_clinvar_hi_match_df['start_outer']
)

nch_clinvar_hi_match_df['jaccard_score_bp_overlap'] = (
    (nch_clinvar_hi_match_df['stop_inner'] - nch_clinvar_hi_match_df['start_inner'])
    / (nch_clinvar_hi_match_df['stop_outer'] - nch_clinvar_hi_match_df['start_outer'])
)

In [72]:
(
    nch_clinvar_hi_match_df
    #.query("jaccard_score_bp_overlap < 0")
    .query("start_38__nch > stop_38__nch")
    #.query("start_38__clinvar > stop_38__clinvar")
)

,variant,build,chromosome,start,stop,start_38__nch,stop_38__nch,copy_number,copy_number_min,copy_number_max,...,absolute_copies,copies,feature_name,start_outer,start_inner,stop_inner,stop_outer,jaccard_score_bp_overlap,intersection_length,union_length
3402,15q11.2(22822019_23085218)x1,GRCh37,15,22822019,23085218,23051047,22787849,1,1,1,...,1.0,1.0,2,22572808,23051047,22787849,23066575,-0.533041,-263198,493767
3404,15q11.2(22822019_23085218)x1,GRCh37,15,22822019,23085218,23051047,22787849,1,1,1,...,1.0,1.0,2,22655581,23051047,22787849,23066575,-0.640394,-263198,410994
3409,15q11.2(22822019_23085218)x1,GRCh37,15,22822019,23085218,23051047,22787849,1,1,1,...,1.0,1.0,4,22358242,23051047,22787849,28481444,-0.042984,-263198,6123202
3411,15q11.2(22822019_23085218)x1,GRCh37,15,22822019,23085218,23051047,22787849,1,1,1,...,1.0,1.0,2,22655581,23051047,22787849,23066575,-0.640394,-263198,410994
3420,15q11.2(22822019_23085218)x1,GRCh37,15,22822019,23085218,23051047,22787849,1,1,1,...,1.0,1.0,2,22572808,23051047,22787849,23066575,-0.533041,-263198,493767
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
902304,"15q11.2(22,971,892-23,034,414)x1",GRCh37,15,22971892,23034414,22901174,22838653,1,1,1,...,1.0,1.0,4,22584720,22901174,22838653,27461850,-0.012819,-62521,4877130
902344,"15q11.2(22,971,892-23,034,414)x1",GRCh37,15,22971892,23034414,22901174,22838653,1,1,1,...,1.0,1.0,4,20534258,22901174,22838653,28321433,-0.008029,-62521,7787175
902354,"15q11.2(22,971,892-23,034,414)x1",GRCh37,15,22971892,23034414,22901174,22838653,1,1,1,...,1.0,1.0,4,22582307,22901174,22838653,28315123,-0.010906,-62521,5732816
902367,"15q11.2(22,971,892-23,034,414)x1",GRCh37,15,22971892,23034414,22901174,22838653,1,1,1,...,1.0,1.0,4,20966970,22901174,22838653,25963714,-0.012512,-62521,4996744


## Matching NCH copy gain CNVs to ClinVar triplosensitivity overlapping variants

In [34]:
# get NCH copy gain CNVs
nch_df_gain = nch_df.query("copy_number_min > 2")
len(nch_df_gain)

4261

In [35]:
#only keep copy gain ClinVar CNVs
clinvar_df_gain = clinvar_df.query("absolute_copies > 2") 
len(clinvar_df_gain)

19330

In [36]:
# cross reference with all TS features
clinvar_ts_df = (
    clinvar_df_gain
    .merge( 
        right=feat_df_ts,
        how='inner',
        left_on='chr',
        right_on='chromosome',
        suffixes=('__clinvar', '__ts')
    )
)
len(clinvar_ts_df)

25191

In [37]:
# only keep total overlap with TS region
clinvar_ts_df = (
    clinvar_ts_df
    .query("start_38 <= genomic_start")
    .query("stop_38 >= genomic_stop")
)
len(clinvar_ts_df)

950

In [38]:
pd.DataFrame(
    clinvar_ts_df
    .groupby(clinvar_df.columns.to_list())
    ['feature_name']
    .nunique()
    .reset_index()
)

,variation_id,name,variation_type,assembly_version,chr,cytogenetic,start_38,stop_38,range_copies,absolute_copies,copies,feature_name
0,32168,GRCh38/hg38 15q11.2-13.1(chr15:23411789-282751...,copy number gain,38,15,15q11.2-13.1,23411788,28275167,[],3.0,3.0,1
1,32239,GRCh37/hg19 17q12(chr17:34611352-36248918)x3,copy number gain,37,17,17q12,238086,37889296,[],3.0,3.0,4
2,32504,GRCh37/hg19 17q12(chr17:34508117-36248918)x3,copy number gain,37,17,17q12,36180740,37889296,[],3.0,3.0,1
3,32998,GRCh38/hg38 22q11.21(chr22:18339130-21207225)x3,copy number gain,38,22,22q11.21,18339129,21207225,[],3.0,3.0,2
4,33584,GRCh38/hg38 15q11.2-13.1(chr15:22358243-284814...,copy number gain,38,15,15q11.2-13.1,22358242,28481444,[],3.0,3.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...
549,3242337,GRCh37/hg19 22q11.1-11.21(chr22:16849364-20311...,copy number gain,37,22,22q11.1-11.21,16368701,20323866,[],3.0,3.0,2
550,3370389,GRCh38/hg38 Xq28(chrX:154348522-154594454)x3,copy number gain,38,X,Xq28,154348521,154594454,[],3.0,3.0,1
551,3391845,GRCh37/hg19 5q35.2-35.3(chr5:175570678-1774145...,copy number gain,37,5,5q35.2-35.3,176143674,177987567,[],3.0,3.0,1
552,3391847,GRCh37/hg19 7q34-36.3(chr7:142491993-159119707)x3,copy number gain,37,7,7q34-36.3,142784182,159327017,[],3.0,3.0,1
